# nnInteractive — GPU Memory Benchmark

Measures peak VRAM on representative 3D volumes using `torch.cuda.max_memory_allocated()` 

(PyTorch-native, equivalent to TF's `get_memory_info()`).



**Size definitions** (by total_voxels = roi_slices × H × W, matching `p_unet_memory.ipynb`):

- **Small**: ≤ 128³ = 2,097,152

- **Medium**: 128³ – 192³

- **Large**: > 192³ = 7,077,888 (nnInteractive AutoZoom threshold)



Same representative volumes as `p_unet_memory.ipynb` for apples-to-apples comparison.


| **§1** | Find representative volumes (same selection logic) |

| **§2** | GPU memory measurement via PyTorch |

| **§3** | Results table (alongside P-UNet values) |

---

## §1 — Find Representative Volumes

In [1]:
import sys
from pathlib import Path

notebook_dir = Path().resolve()
project_root = notebook_dir.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
from evaluation.benchmark_nninteractive.memory_utils import (
    scan_from_pkl, pick_largest, load_volume_for_memory,
)

In [2]:
SMALL_LIMIT = 128 ** 3   # 2,097,152
LARGE_LIMIT = 192 ** 3   # 7,077,888 (nnInteractive AutoZoom)

NPZ_PATHS = [
    project_root / 'data' / 'test_data' / 'FLARE_2022.npz',
    project_root / 'data' / 'test_data' / 'han_seg_ct.npz',
    project_root / 'data' / 'test_data' / 'han_seg_mri.npz',
    project_root / 'data' / 'test_data' / 'HCCTase_ceCT.npz',
    project_root / 'data' / 'test_data' / 'SegRap2023.npz',
    project_root / 'data' / 'test_data' / 'TotalSeg_mri.npz',
]

# Benchmark results pkl — same population as p_unet_memory.ipynb
PKL_PATH = "results_lisp-net_ssf_none_ifl_ssf_ConfidenceDrop_df0.1_20260720_173154.pkl"

candidates = scan_from_pkl(PKL_PATH, NPZ_PATHS, small_limit=SMALL_LIMIT, large_limit=LARGE_LIMIT)

from collections import Counter
print(f'\nSize distribution (by total_voxels):')
for label, count in Counter(c['size_bin'] for c in candidates).most_common():
    print(f'  {label}: {count}')

Indexing NPZ files ...
  362 patients indexed
Scanned 1432 runs from pkl (1368 unique volume/ROI/axis configurations)

Size distribution (by total_voxels):
  Medium: 530
  Large: 472
  Small: 430


In [3]:
rep = {}
for label in ['Small', 'Medium', 'Large']:
    best = pick_largest(candidates, label)
    if best:
        rep[label] = best
        print(f"{label}: {best['dataset_name']}/{best['pid']} roi={best['roi']} axis={best['axis']} roi_h={best['roi_h']} roi_w={best['roi_w']} roi_area={best['roi_h']*best['roi_w']:,}")

Small: TotalSeg_mri/s0131 roi=15 axis=2 roi_h=255 roi_w=69 roi_area=17,595
Medium: TotalSeg_mri/s0153 roi=11 axis=2 roi_h=419 roi_w=100 roi_area=41,900
Large: TotalSeg_mri/s0245 roi=5 axis=1 roi_h=276 roi_w=319 roi_area=88,044


## §2 — GPU Memory Measurement (PyTorch / nnInteractive)

In [4]:
import torch
from evaluation.benchmark_nninteractive.nninteractive_inference import NNInteractiveInference

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB')

/software/anaconda3/envs/machauer/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda:0
  GPU: NVIDIA RTX 6000 Ada Generation
  VRAM total: 47.4 GB


In [5]:
def profile_nninteractive(rep):
    """Run nnInteractive inference and return peak VRAM (PyTorch-native).

    Uses torch.cuda.max_memory_allocated() which reports the peak memory
    used by PyTorch ops (equivalent to TF's get_memory_info()['peak']).

    Warm-up run absorbs one-time model-loading / JIT allocations, then we
    reset the peak counter and measure the steady-state inference memory.
    """
    img_3d, seg_3d_binary, prompt_2d, prompt_idx = load_volume_for_memory(rep)

    # Build initial_prompt_3d: 3D binary mask with the prompt slice set
    initial_prompt_3d = np.zeros_like(img_3d, dtype=np.float32)
    idx = [slice(None)] * 3
    idx[rep['axis']] = prompt_idx
    initial_prompt_3d[tuple(idx)] = prompt_2d

    # nnInteractive expects (1, X, Y, Z) with leading channel dim
    img_4d = np.expand_dims(img_3d, axis=0)

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    infer = NNInteractiveInference()

    # Warm-up — absorbs one-shot model loading / compilation
    _ = infer.run(
        img_4d=img_4d,
        seg_3d=seg_3d_binary,
        initial_prompt_3d=initial_prompt_3d,
        user_interacts_idx=[],
        prompt_axis=rep['axis'],
        prompt_idx=prompt_idx,
    )

    # Reset peak stats to post-warmup baseline, then measure steady-state
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(DEVICE)

    _ = infer.run(
        img_4d=img_4d,
        seg_3d=seg_3d_binary,
        initial_prompt_3d=initial_prompt_3d,
        user_interacts_idx=[],
        prompt_axis=rep['axis'],
        prompt_idx=prompt_idx,
    )

    peak_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 * 1024)
    return peak_mb

In [6]:
results = {}
for label in ['Small', 'Medium', 'Large']:
    if label not in rep:
        continue
    r = rep[label]
    print(
        f"\nProfiling {label} volume: "
        f"{r['pid']} (roi={r['roi']}, axis={r['axis']})"
    )
    peak_mb = profile_nninteractive(r)
    results[label] = peak_mb
    print(f'  Peak VRAM (PyTorch ops): {peak_mb:.1f} MB')


Profiling Small volume: s0131 (roi=15, axis=2)
[NNInteractiveInference] Downloading nnInteractive_v1.0 from nnInteractive/nnInteractive …


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.
[NNInteractiveInference] Initialising session on device=cuda:0 …
License reminder: The official nnInteractive checkpoint is licensed under Creative Commons Attribution Non Commercial Share Alike 4.0 (CC BY-NC-SA 4.0). See the license note in readme.md (# License).
[NNIn

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

[NNInteractiveInference] Initialising session on device=cuda:0 …
License reminder: The official nnInteractive checkpoint is licensed under Creative Commons Attribution Non Commercial Share Alike 4.0 (CC BY-NC-SA 4.0). See the license note in readme.md (# License).
[NNInteractiveInference] Ready.
Added new image interaction: scale 2.515625, center [[228, 50, 186]]
Took 0.252 s for initial prediction at zoom out factor 2.515625
Zoom out took 0.0 s, max zoom out factor 2.515625
Forcing full refinement of entire structure
Took 0.575 s for refining the segmentation with 5 bounding boxes
Done. Total time 0.86s
Added new image interaction: scale 2.515625, center [[228, 50, 186]]
Took 0.279 s for initial prediction at zoom out factor 2.515625
Zoom out took 0.0 s, max zoom out factor 2.515625
Forcing full refinement of entire structure
Took 0.792 s for refining the segmentation with 5 bounding boxes
Done. Total time 1.104s
  Peak VRAM (PyTorch ops): 4209.9 MB

Profiling Large volume: s0245 (roi

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

[NNInteractiveInference] Initialising session on device=cuda:0 …
License reminder: The official nnInteractive checkpoint is licensed under Creative Commons Attribution Non Commercial Share Alike 4.0 (CC BY-NC-SA 4.0). See the license note in readme.md (# License).
[NNInteractiveInference] Ready.
Added new image interaction: scale 1.8645833333333333, center [[214, 138, 214]]
Took 0.252 s for initial prediction at zoom out factor 1.8645833333333333
AutoZoom zoom out factor 1.8645833333333333
Zoom out took 0.432 s, max zoom out factor 2.796875
Forcing full refinement of entire structure
Took 1.095 s for refining the segmentation with 6 bounding boxes
Done. Total time 1.855s
Added new image interaction: scale 1.8645833333333333, center [[214, 138, 214]]
Took 0.23 s for initial prediction at zoom out factor 1.8645833333333333
AutoZoom zoom out factor 1.8645833333333333
Zoom out took 0.391 s, max zoom out factor 2.796875
Forcing full refinement of entire structure
Took 1.144 s for refining t